# 1. Setup: Packages and Global Parameters


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import requests

URL = "https://incidentdatabase.ai/api/graphql"

introspection_query = """
{
  __type(name: "Incident") {
    name
    fields {
      name
      type { name kind ofType { name kind } }
    }
  }
}
"""

# The AIID GraphQL endpoint rejects POST requests whose Origin/Referer header
# isn't https://incidentdatabase.ai or https://staging-aiid.netlify.app
# (see site/gatsby-site/netlify/functions/graphql.ts in responsible-ai-collaborative/aiid).
# That check only applies to POST — GET requests bypass it, so we send the
# query as a URL parameter instead.
r = requests.get(URL, params={"query": introspection_query})
print(r.json())

In [ ]:
import csv
import glob
import json
import os

# Folder in Google Drive containing the batch JSON files to combine.
BATCH_DIR = "/content/drive/MyDrive/phd/P4/Data"
BATCH_PATTERN = "batch*.json"
OUTPUT_CSV = os.path.join(BATCH_DIR, "aiid_full_incidents_processed.csv")

FIELDNAMES = [
    "incident_id", "title", "description", "date", "date_modified", "year",
    "developer", "deployer", "harmed", "implicated_systems", "report_count",
    "mit_risk_domain", "mit_risk_subdomain", "mit_entity", "mit_intent", "mit_timing",
    "cset_sector", "cset_lives_lost", "cset_injuries", "cset_location_country",
    "cset_ai_system_description", "cset_entities_raw",
    "gmf_known_ai_goal", "gmf_known_technology", "gmf_known_technical_failure",
    "n_cset_annotators",
]


def names_join(items):
    return "; ".join(x.get("name", "") for x in (items or []) if x)


def find_classification(classifications, namespace, require_publish=False):
    for c in classifications or []:
        if c.get("namespace") == namespace and (not require_publish or c.get("publish")):
            return c
    return None


def count_namespace_prefix(classifications, prefix):
    return sum(1 for c in classifications or [] if str(c.get("namespace", "")).startswith(prefix))


def classification_attr(classification, short_name):
    """Look up an attribute by short_name and decode its JSON-encoded value_json."""
    if not classification:
        return None
    for a in classification.get("attributes", []):
        if a.get("short_name") == short_name:
            try:
                return json.loads(a["value_json"])
            except (TypeError, ValueError):
                return a["value_json"]
    return None


def build_row(incident):
    classifications = incident.get("classifications") or []
    mit = find_classification(classifications, "MIT")
    csetv1 = find_classification(classifications, "CSETv1")
    # GMF ("known AI goal/technology/failure") is only trustworthy once published.
    gmf = find_classification(classifications, "GMF", require_publish=True)

    date = incident.get("date") or ""
    sector = classification_attr(csetv1, "Sector of Deployment")
    lives_lost = classification_attr(csetv1, "Lives Lost")
    injuries = classification_attr(csetv1, "Injuries")

    return {
        "incident_id": incident.get("incident_id"),
        "title": incident.get("title"),
        "description": incident.get("description"),
        "date": date,
        "date_modified": incident.get("date_modified") or "",
        "year": date[:4] if date else "",
        "developer": names_join(incident.get("AllegedDeveloperOfAISystem")),
        "deployer": names_join(incident.get("AllegedDeployerOfAISystem")),
        "harmed": names_join(incident.get("AllegedHarmedOrNearlyHarmedParties")),
        "implicated_systems": names_join(incident.get("implicated_systems")),
        "report_count": len(incident.get("reports") or []),
        "mit_risk_domain": classification_attr(mit, "Risk Domain") or "",
        "mit_risk_subdomain": classification_attr(mit, "Risk Subdomain") or "",
        "mit_entity": classification_attr(mit, "Entity") or "",
        "mit_intent": classification_attr(mit, "Intent") or "",
        "mit_timing": classification_attr(mit, "Timing") or "",
        "cset_sector": "" if sector is None else str(sector),
        "cset_lives_lost": float(lives_lost) if isinstance(lives_lost, (int, float)) else "",
        "cset_injuries": float(injuries) if isinstance(injuries, (int, float)) else "",
        "cset_location_country": classification_attr(csetv1, "Location Country (two letters)") or "",
        "cset_ai_system_description": classification_attr(csetv1, "AI System Description") or "",
        # csetv1 present but attribute missing/null -> "null"; csetv1 absent entirely -> "".
        "cset_entities_raw": json.dumps(classification_attr(csetv1, "Entities")) if csetv1 is not None else "",
        "gmf_known_ai_goal": json.dumps(classification_attr(gmf, "Known AI Goal")) if gmf is not None else "",
        "gmf_known_technology": json.dumps(classification_attr(gmf, "Known AI Technology")) if gmf is not None else "",
        "gmf_known_technical_failure": json.dumps(classification_attr(gmf, "Known AI Technical Failure")) if gmf is not None else "",
        # Count of CSETv1_Annotator-* classification blocks, published or not.
        "n_cset_annotators": count_namespace_prefix(classifications, "CSETv1_Annotator-"),
    }


batch_paths = sorted(glob.glob(os.path.join(BATCH_DIR, BATCH_PATTERN)))
print(f"Found {len(batch_paths)} batch file(s)")

incidents_by_id = {}

for path in batch_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Each batch file is a raw GraphQL response: {"data": {"incidents": [...]}}.
    records = next(iter(data["data"].values())) if isinstance(data, dict) and "data" in data else data

    for incident in records:
        incidents_by_id[incident["incident_id"]] = incident

    print(f"  {os.path.basename(path)}: {len(records)} record(s)")

print(f"Total unique incidents: {len(incidents_by_id)}")

rows = [build_row(inc) for inc in incidents_by_id.values()]
rows.sort(key=lambda r: r["incident_id"])

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows to {OUTPUT_CSV}")

# 2. Descriptive Analysis


In [ ]:
import pandas as pd

CSV_PATH = "/content/drive/MyDrive/phd/P4/Data/aiid_full_incidents_processed.csv"

df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print()
print("Columns and dtypes:")
print(df.dtypes)

In [ ]:
# Missing values per column
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0]

In [ ]:
# Summary statistics for numeric columns
df.describe(include="number").T

In [ ]:
# Value counts for object/categorical columns with manageable cardinality
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    n_unique = df[col].nunique(dropna=True)
    if 1 < n_unique <= 30:
        print(f"--- {col} ({n_unique} unique values) ---")
        print(df[col].value_counts(dropna=False).head(10))
        print()

In [ ]:
# Incidents over time, if a date-like column is present (e.g. "date")
date_cols = [c for c in df.columns if "date" in c.lower()]

if date_cols:
    date_col = date_cols[0]
    dates = pd.to_datetime(df[date_col], errors="coerce")
    print(f"Using date column: {date_col}")
    print(f"Range: {dates.min()} to {dates.max()}")
    print()
    print("Incidents per year:")
    print(dates.dt.year.value_counts().sort_index())
else:
    print("No date-like column found.")